# Huấn luyện ViSoBERT / PhoBERT nhận diện cảm xúc (UIT-VSMEC) cho HuTube — v3
Thay đổi so với bản trước:
- **Công tắc A/B cho tiền xử lý** (`PREPROC_MODE`): `raw` / `light` / `teencode`, để đo xem chuẩn hóa có thật sự giúp hay không.
- **Teencode an toàn hơn**: không còn đổi `minh`, `dậy`, `vcl`, `đéo`, `tao`... (những từ làm hỏng nghĩa hoặc xóa tín hiệu cảm xúc).
- **Emoji**: giữ nguyên với ViSoBERT, chỉ chuyển sang chữ với PhoBERT (tự động, có thể ghi đè).
- **Ngưỡng kiểm duyệt chọn trên tập validation**, sau đó mới báo cáo trên test (không rò rỉ).
- **Padding động** (nhanh hơn), `seed` cố định, lưu xác suất để **ensemble nhiều seed**.
- Lưu `inference_config.json` cạnh model để emotion-service dùng đúng cấu hình lúc train.

**Cách chạy thí nghiệm đối chứng:** chạy lần lượt `PREPROC_MODE = "raw"`, `"light"`, `"teencode"` (cùng seed), so sánh `f1_macro` trên **validation**. Sau đó chạy chế độ tốt nhất với 3 seed (42, 43, 44) và lấy trung bình.

In [ ]:
# Bước 1: Cài đặt thư viện
!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn emoji pyvi

In [ ]:
# Bước 2: CẤU HÌNH (chỉ cần sửa ở đây)
MODEL_NAME = "uitnlp/visobert"      # hoặc "vinai/phobert-base-v2"
PREPROC_MODE = "light"              # "raw" | "light" | "teencode"
DEMOJIZE = None                     # None = tự động (True nếu PhoBERT, False nếu ViSoBERT)
SEED = 42
MAX_LEN = 128
EPOCHS = 8
LR = 3e-5
USE_CLASS_WEIGHTS = True
TARGET_TOXIC_RECALL = 0.85          # mục tiêu recall khi chọn ngưỡng trên validation

IS_PHOBERT = "phobert" in MODEL_NAME.lower()
if DEMOJIZE is None:
    DEMOJIZE = IS_PHOBERT
TAG = f"{MODEL_NAME.split('/')[-1]}_{PREPROC_MODE}_seed{SEED}"
print("Cấu hình:", dict(MODEL=MODEL_NAME, PREPROC=PREPROC_MODE, DEMOJIZE=DEMOJIZE, SEED=SEED, TAG=TAG))

In [ ]:
# Bước 3: Tải dữ liệu UIT-VSMEC
import urllib.request, json

BASE_URL = "https://huggingface.co/datasets/tridm/UIT-VSMEC/raw/main"

def fetch_data(split_name, filename):
    url = f"{BASE_URL}/{filename}"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    print(f"[*] Đã tải {split_name}: {len(data)} mẫu")
    return data

train_data = fetch_data("train", "train.json")
val_data = fetch_data("validation", "valid.json")
test_data = fetch_data("test", "test.json")

In [ ]:
# Bước 4: Tiền xử lý (có công tắc A/B)
import re, unicodedata, emoji
from pyvi import ViTokenizer

# Chỉ giữ các viết tắt rõ nghĩa, KHÔNG đụng vào từ chửi (vcl, đéo...) vì đó là tín hiệu Anger/Disgust
SAFE_TEENCODE = {
    r"\b(k|ko|kh|hok|hong)\b": "không",
    r"\b(dc|đc)\b": "được",
    r"\b(nhiu|nhìu)\b": "nhiều",
    r"\b(mk|mik)\b": "mình",
    r"\b(cx)\b": "cũng",
    r"\b(vs)\b": "với",
    r"\b(wa|wá)\b": "quá",
    r"\b(tks|thanks)\b": "cảm ơn",
}
_TEEN = [(re.compile(p, re.IGNORECASE), r) for p, r in SAFE_TEENCODE.items()]

def clean_social_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    if DEMOJIZE:
        text = emoji.demojize(text, language="alias")
        text = re.sub(r":([a-zA-Z0-9_]+):", lambda m: " " + m.group(1).replace("_", " ") + " ", text)
    if PREPROC_MODE == "teencode":            # làm trước để "kkk" không bị biến thành "không"
        for pat, rep in _TEEN:
            text = pat.sub(rep, text)
    if PREPROC_MODE in ("light", "teencode"):  # quáááá -> quá
        text = re.sub(r"([a-zA-ZÀ-ỹ])\1{2,}", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    if IS_PHOBERT:                             # PhoBERT cần văn bản đã tách từ
        text = ViTokenizer.tokenize(text)
    return text

for s in ["Video xàm vcl quáááá 😡😡 ad làm ăn như kẹcccc ko xem nữa",
          "Anh minh bạch thông minh, thức dậy sớm kkk"]:
    print("Trước:", s)
    print("Sau  :", clean_social_text(s), "\n")

In [ ]:
# Bước 5: Nhãn, ánh xạ sắc thái, trọng số lớp
import numpy as np, torch
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight

EMOTION_LABELS = ["Enjoyment", "Disgust", "Sadness", "Anger", "Fear", "Surprise", "Other"]
LABEL2ID = {l: i for i, l in enumerate(EMOTION_LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

EMOTION_VI = {
    "Enjoyment": "Thích thú / Vui vẻ", "Disgust": "Ghê tởm / Phản cảm",
    "Sadness": "Buồn bã / Thất vọng", "Anger": "Tức giận / Chửi bới",
    "Fear": "Sợ hãi / Lo lắng", "Surprise": "Ngạc nhiên / Bất ngờ",
    "Other": "Bình thường / Khác",
}
SENTIMENT_VI = {"POSITIVE": "Tích cực", "NEGATIVE": "Tiêu cực", "NEUTRAL": "Trung tính"}
EMOTION_TO_SENTIMENT = {
    "Enjoyment": "POSITIVE", "Disgust": "NEGATIVE", "Sadness": "NEGATIVE",
    "Anger": "NEGATIVE", "Fear": "NEGATIVE", "Surprise": "NEUTRAL", "Other": "NEUTRAL",
}
ANGER_IDX, DISGUST_IDX = LABEL2ID["Anger"], LABEL2ID["Disgust"]

y_train = np.array([LABEL2ID[d["Emotion"]] for d in train_data])
w_sqrt = compute_class_weight("balanced", classes=np.arange(7), y=y_train) ** 0.5
CLASS_WEIGHTS = torch.tensor(w_sqrt, dtype=torch.float)

print("Phân bố train & trọng số loss:")
cnt = Counter(y_train.tolist())
for i, e in enumerate(EMOTION_LABELS):
    print(f"  {e:10} n={cnt[i]:5}  weight={w_sqrt[i]:.3f}")

In [ ]:
# Bước 6: Tokenizer + Dataset (tokenize trước, padding động khi train)
from torch.utils.data import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding, set_seed

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=not IS_PHOBERT)

class VSMECDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        texts = [clean_social_text(d["Sentence"]) for d in data]
        enc = tokenizer(texts, truncation=True, max_length=max_length)
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = [LABEL2ID[d["Emotion"]] for d in data]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {"input_ids": self.input_ids[i],
                "attention_mask": self.attention_mask[i],
                "labels": self.labels[i]}

train_ds = VSMECDataset(train_data, tokenizer, MAX_LEN)
val_ds = VSMECDataset(val_data, tokenizer, MAX_LEN)
test_ds = VSMECDataset(test_data, tokenizer, MAX_LEN)
collator = DataCollatorWithPadding(tokenizer)

lens = [len(x) for x in train_ds.input_ids]
print(f"[+] train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} | độ dài token trung bình={np.mean(lens):.1f}, max={max(lens)}")

In [ ]:
# Bước 7: Huấn luyện (loss có trọng số + label smoothing)
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(EMOTION_LABELS), id2label=ID2LABEL, label2id=LABEL2ID
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        w = CLASS_WEIGHTS.to(out.logits.device) if USE_CLASS_WEIGHTS else None
        loss = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)(out.logits, labels)
        return (loss, out) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir=f"./outputs/{TAG}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=150,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"[*] Bắt đầu huấn luyện: {TAG}")
trainer.train()
val_metrics = trainer.evaluate(val_ds)
print("\n>>> VALIDATION:", {k: round(v, 4) for k, v in val_metrics.items() if k.startswith("eval_") and k != "eval_runtime"})

In [ ]:
# Bước 8: Đánh giá trên validation và test, lưu xác suất để ensemble
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix

def softmax_np(x):
    return F.softmax(torch.tensor(x), dim=-1).numpy()

val_out = trainer.predict(val_ds)
test_out = trainer.predict(test_ds)
val_probs, test_probs = softmax_np(val_out.predictions), softmax_np(test_out.predictions)
val_labels, labels = val_out.label_ids, test_out.label_ids
preds = test_probs.argmax(-1)

np.save(f"probs_val_{TAG}.npy", val_probs)     # dùng để ensemble: trung bình các file probs_* của nhiều seed
np.save(f"probs_test_{TAG}.npy", test_probs)

print("=== TEST: 7 CẢM XÚC ===")
print(classification_report(labels, preds, target_names=[f"{e} ({EMOTION_VI[e]})" for e in EMOTION_LABELS], zero_division=0, digits=3))

to_sent = lambda ids: [SENTIMENT_VI[EMOTION_TO_SENTIMENT[ID2LABEL[i]]] for i in ids]
true_sents, pred_sents = to_sent(labels), to_sent(preds)
print("\n=== TEST: SẮC THÁI ===")
print(classification_report(true_sents, pred_sents, zero_division=0, digits=3))

In [ ]:
# Bước 9: Chọn ngưỡng kiểm duyệt TRÊN VALIDATION, rồi báo cáo trên TEST
# Lưu ý: "độc hại" ở đây = Anger hoặc Disgust (nhãn thay thế, không phải nhãn toxic thật)
from sklearn.metrics import precision_recall_fscore_support

is_toxic = lambda ids: np.array([1 if ID2LABEL[i] in ("Anger", "Disgust") else 0 for i in ids])
toxic_score = lambda probs: probs[:, ANGER_IDX] + probs[:, DISGUST_IDX]

y_val_t, s_val = is_toxic(val_labels), toxic_score(val_probs)
y_test_t, s_test = is_toxic(labels), toxic_score(test_probs)

rows = []
for th in np.arange(0.10, 0.76, 0.05):
    p, r, f, _ = precision_recall_fscore_support(y_val_t, (s_val > th).astype(int), average="binary", zero_division=0)
    rows.append((round(float(th), 2), p, r, f))

print("Quét ngưỡng trên VALIDATION:")
print("  ngưỡng | precision | recall | f1")
for th, p, r, f in rows:
    print(f"   {th:.2f}  |   {p:.3f}   | {r:.3f}  | {f:.3f}")

ok = [row for row in rows if row[2] >= TARGET_TOXIC_RECALL]
THRESHOLD = max(ok, key=lambda x: x[0])[0] if ok else max(rows, key=lambda x: x[3])[0]
print(f"\n[+] Ngưỡng được chọn: {THRESHOLD} "
      f"({'đạt' if ok else 'KHÔNG đạt'} recall mục tiêu {TARGET_TOXIC_RECALL} trên validation)")

y_pred_t = (s_test > THRESHOLD).astype(int)
print("\n=== TEST: KIỂM DUYỆT ĐỘC HẠI (ngưỡng chọn từ validation) ===")
print(classification_report(y_test_t, y_pred_t, target_names=["An toàn (visible)", "Độc hại (held)"], digits=3))

In [ ]:
# Bước 10: Ma trận nhầm lẫn
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
cm_labels = [f"{e}\n({EMOTION_VI[e].split('/')[0].strip()})" for e in EMOTION_LABELS]
sns.heatmap(confusion_matrix(labels, preds), annot=True, fmt="d", cmap="Blues",
            xticklabels=cm_labels, yticklabels=cm_labels, ax=axes[0])
axes[0].set_title(f"7 cảm xúc — {TAG}", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Dự đoán"); axes[0].set_ylabel("Thực tế")

names = ["An toàn", "Độc hại"]
sns.heatmap(confusion_matrix(y_test_t, y_pred_t), annot=True, fmt="d", cmap="Reds",
            xticklabels=names, yticklabels=names, ax=axes[1])
axes[1].set_title(f"Kiểm duyệt độc hại (ngưỡng {THRESHOLD})", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Dự đoán"); axes[1].set_ylabel("Thực tế")

plt.tight_layout()
plt.savefig(f"evaluation_{TAG}.png", dpi=200)
plt.show()

In [ ]:
# Bước 11: Thử nghiệm với bình luận thực tế
demo_comments = [
    "Video hướng dẫn siêu có tâm, cảm ơn anh nhiều ạ! ❤️👍",
    "Video xàm vcl lừa đảo quááá, cút đi ad ngu ngốc 😡🤬",
    "Kênh làm chán quá, thất vọng ghê haizzz",
    "Cho mik xin tên bài nhạc ở phút thứ 2 với bạn ơi",
    "Trời ơi xem đoạn cuối bất ngờ thực sự ko tin nổi luôn 😱",
]

model.eval()
for cmt in demo_comments:
    clean = clean_social_text(cmt)
    inputs = tokenizer(clean, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
    with torch.no_grad():
        p = F.softmax(model(**inputs).logits, dim=-1).cpu().numpy()[0]
    best = int(p.argmax())
    emo = ID2LABEL[best]
    p_toxic = float(p[ANGER_IDX] + p[DISGUST_IDX])
    status = f"held (độc hại {p_toxic*100:.1f}%)" if p_toxic > THRESHOLD else f"visible (độc hại {p_toxic*100:.1f}%)"
    print(f"Bình luận : {cmt}")
    print(f"  Làm sạch: {clean}")
    print(f"  Cảm xúc : {emo} - {EMOTION_VI[emo]} ({p[best]*100:.1f}%)")
    print(f"  Sắc thái: {SENTIMENT_VI[EMOTION_TO_SENTIMENT[emo]]}")
    print(f"  HuTube  : {status}\n")

In [ ]:
# Bước 12: Lưu model + cấu hình suy luận, nén và tải về
import json, os

SAVE_NAME = f"{MODEL_NAME.split('/')[-1].replace('-', '_')}_vsmec_best"
trainer.save_model(SAVE_NAME)
tokenizer.save_pretrained(SAVE_NAME)

inference_config = {
    "model_name": MODEL_NAME,
    "preproc_mode": PREPROC_MODE,
    "demojize": DEMOJIZE,
    "use_pyvi_word_segmentation": IS_PHOBERT,
    "max_len": MAX_LEN,
    "toxic_labels": ["Anger", "Disgust"],
    "toxic_threshold": THRESHOLD,
    "teencode_dict": SAFE_TEENCODE if PREPROC_MODE == "teencode" else {},
    "emotion_to_sentiment": EMOTION_TO_SENTIMENT,
}
with open(os.path.join(SAVE_NAME, "inference_config.json"), "w", encoding="utf-8") as f:
    json.dump(inference_config, f, ensure_ascii=False, indent=2)

!zip -qr {SAVE_NAME}.zip {SAVE_NAME}
from google.colab import files
files.download(f"{SAVE_NAME}.zip")
print(f"[+] Đã tạo {SAVE_NAME}.zip")
print(f"Giải nén vào: emotion-service/saved_models/{SAVE_NAME}/")
print("Service phải: (1) dùng đúng clean_social_text như trên, (2) đọc inference_config.json, (3) dùng tokenizer lưu cùng model.")

In [ ]:
# (Tùy chọn) Bước 13: Ensemble nhiều seed — chạy sau khi đã train xong 3 seed (42, 43, 44) và copy các file probs_*.npy vào cùng thư mục
import glob
files_val = sorted(glob.glob("probs_val_*.npy"))
files_test = sorted(glob.glob("probs_test_*.npy"))
print("Tìm thấy:", files_test)
if len(files_test) >= 2:
    ens_val = np.mean([np.load(f) for f in files_val], axis=0)
    ens_test = np.mean([np.load(f) for f in files_test], axis=0)
    ens_preds = ens_test.argmax(-1)
    print(f"Ensemble {len(files_test)} mô hình — test accuracy={accuracy_score(labels, ens_preds):.4f}, "
          f"f1_macro={f1_score(labels, ens_preds, average='macro'):.4f}")
    print("Lưu ý: chỉ nên ensemble các file cùng PREPROC_MODE và cùng MODEL_NAME.")